# 5.1 回归任务：糖尿病数据集

线性回归、随机森林回归、XGBoost 回归；MSE、R²；预测值 vs 真实值散点图。

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 1. 数据加载与探索性分析

In [ ]:
diabetes = load_diabetes()
diabetes_df = pd.DataFrame(data=diabetes.data, columns=diabetes.feature_names)
diabetes_df["target"] = diabetes.target

print("数据集形状：", diabetes_df.shape)
print("缺失值：", diabetes_df.isnull().sum().sum())

X = diabetes_df.drop("target", axis=1)
y = diabetes_df["target"]

correlation_matrix = diabetes_df.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    center=0,
    vmin=-1,
    vmax=1,
)
plt.title("糖尿病数据集特征相关性热力图")
plt.tight_layout()
plt.show()

## 2. 预处理：标准化与划分训练/测试集

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("标准化前特征均值（第1列）：", float(X.iloc[:, 0].mean()))
print("标准化后特征均值（第1列）：", round(float(X_scaled[:, 0].mean()), 2))

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)
print("训练集样本数：", X_train.shape[0])
print("测试集样本数：", X_test.shape[0])

## 3. 模型训练

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
print("线性回归训练完成")

rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
print("随机森林回归训练完成")

xgb_reg = xgb.XGBRegressor(random_state=42, objective="reg:squarederror")
xgb_reg.fit(X_train, y_train)
print("XGBoost 回归训练完成")

## 4. 评估与可视化

In [ ]:
def evaluate_model(model, model_name: str):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"\n{model_name}")
    print(f"  MSE: {mse:.4f}")
    print(f"  R2:  {r2:.4f}")
    return y_pred


y_pred_lr = evaluate_model(lr, "线性回归")
y_pred_rf = evaluate_model(rf, "随机森林回归")
y_pred_xgb = evaluate_model(xgb_reg, "XGBoost 回归")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models_preds = [
    ("线性回归", y_pred_lr, "steelblue"),
    ("随机森林回归", y_pred_rf, "green"),
    ("XGBoost 回归", y_pred_xgb, "orange"),
]
for ax, (name, y_pred, color) in zip(axes, models_preds):
    ax.scatter(y_test, y_pred, alpha=0.6, color=color)
    lo, hi = float(y_test.min()), float(y_test.max())
    ax.plot([lo, hi], [lo, hi], "r--")
    ax.set_xlabel("真实病情指标")
    ax.set_ylabel("预测病情指标")
    ax.set_title(f"{name}：预测 vs 真实")
plt.tight_layout()
plt.show()